In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent

In [ ]:
loader=PyPDFLoader("data/GEN_AI_2026.pdf")
docs=loader.load()
len(docs)

In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_docs=splitter.split_documents(docs)

In [ ]:
len(splitted_docs)

In [ ]:
embeddings=OpenAIEmbeddings(model="text-embedding-3-small",chunk_size=1000)
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings)

In [ ]:
same_records=vector_store.similarity_search("What is what is trading ?",k=2)

In [ ]:
same_records[0].page_content

Agent Build , to need tools llm  prompt

In [ ]:
@tool
def retriever_tool(query:str):
    """
    this tool can help you to retrieve the relevant data of the pdf document, and these documents have details about school reports
 
    """
    print("tool called:",query)
    docs=vector_store.similarity_search(query,k=6)
    context=""
    for doc in docs:
        context+=doc.page_content+"\n\n"
   

In [ ]:
retriever_tool("What is science teacher name ?")

In [ ]:
llm=ChatOpenAI(model="gpt-5")


In [ ]:
System_Prompt="""
you are a helpful assistant for school report, you can use retriever_tool to retrieve the relevant data of the pdf document, and these documents have details about school reports
"""

In [ ]:
agent=create_agent(llm=llm,tools=[retriever_tool],system_prompt=System_Prompt)

In [ ]:
query="What is science teacher name ?"
response=agent.invoke({"message":[{"role":"user","content":query}]})


In [ ]:
result=response["message"][-1].content

In [ ]:
print(result)